## Example: Build Quality Controlled MADIS Dataset
This notebook demonstrates how to build a quality-controlled MADIS
dataset using HRRR data as the reference dataset in the quality control process.
It reads in MADIS and HRRR data, merges them, applies quality control filters,
and saves the cleaned dataset as a NetCDF file for an individual year.

This example runs on 9 stations, one from each of NOAA's National Centers
for Environmental Information (NCEI) US Climate Regions (Karl and Koss 1984),
to demonstrate the workflow on a small dataset.

### Data Requirements
- MADIS data in NetCDF format with dimensions `(time, space)`
- HRRR data in NetCDF format with dimensions `(time, x, y)`
- US states shapefile with "STUSPS" column for state abbreviations 
  (This is optional for the QC process but required to run the example notebook unmodified.)


## Inputs

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
from concurrent.futures import ProcessPoolExecutor, as_completed

from madis.qaqc.preprocessing import calc_windspeed, calc_winddirection, \
    read_madis, subset_grid_to_point_xy, forecast_obs_merge, add_abs_diff
from madis.qaqc.configs import MASK_DEPENDENCIES, CLEAN_ORDER, CONFIG_MAP
from madis.qaqc.process import  process_region, get_region_ds

## User Configuration

In [ ]:
# If year in filepaths
year = 2021

In [ ]:
# Update these to point to your local MADIS and HRRR data directories
madis_dir = "path/to/your/madis/"
forecast_dir = "path/to/your/hrrr/"
output_dir = "path/to/your/output/"

madis_path = f"{madis_dir}/*{year}.nc"
forecast_path = f"{forecast_dir}/*{year}.nc"
output_path = f"{output_dir}/qc_madis_{year}.nc"

# Required to quality control and parallelize the dataset by region
shape_file_path = "path/to/your/shapefile.shp"

In [ ]:
# Dictionary pairing MADIS variable names to the corresponding HRRR var names.
VAR_NAME_MAP = {
    "windspeed_10m": "hrrr_ws_10",
    "winddirection_10m": "hrrr_winddir_10",
    "windgust_10m": "hrrr_gust_2m",
    "temperature_2m": "hrrr_t2m",
    "relativehumidity_2m": "hrrr_r2",
}

In [ ]:
# Dictionary defining regions to subset the dataset to.
# States shapefile must contain a column named 'STUSPS' with the state codes.
CLIMATE_REGIONS = {
    "Northeast": ["CT", "DE", "ME", "MD", "MA", "NH", "NJ",
                  "NY", "PA", "RI", "VT"],
    "Upper_Midwest": ["IA", "MI", "MN", "WI"],
    "Ohio_Valley": ["IL", "IN", "KY", "MO", "OH", "TN", "WV"],
    "Southeast": ["AL", "FL", "GA", "NC", "SC", "VA"],
    "N_Rockies": ["MT", "NE", "ND", "SD", "WY"],
    "South": ["AR", "KS", "LA", "MS", "OK", "TX"],
    "Southwest": ["AZ", "CO", "NM", "UT"],
    "Northwest": ["ID", "OR", "WA"],
    "West": ["CA", "NV"]
}

## Read in and prepare data
Read in MADIS and HRRR data, subset to the region of interest, and merge the datasets based on time and location.

In [ ]:
ds_obs = read_madis(madis_path)
ds_obs = ds_obs.unify_chunks()

In [ ]:
hrrr = xr.open_dataset(
    forecast_path,
    engine='netcdf4',
    chunks={}
)
hrrr = hrrr.assign_coords(time=pd.to_datetime(hrrr['time'].values))

In [ ]:
# DO NOT RUN THIS CELL TO RUN QUALITY CONTROL ON THE FULL DATASET!
#Subset to the 9 target points for each of the
# 9 NOAA US Climate Regions (Karl and Koss 1984)
#if use of the full dataset is desired, remove this block.

REGION_TARGET_POINTS = {
    "Northeast":     (42.3601, -71.0589),   # Boston, MA
    "Upper_Midwest": (44.9778, -93.2650),   # Minneapolis, MN
    "Ohio_Valley":   (39.9612, -82.9988),   # Columbus, OH
    "Southeast":     (33.7490, -84.3880),   # Atlanta, GA
    "N_Rockies":     (46.8797, -110.3626),  # Helena, MT (central MT)
    "South":         (35.4676, -97.5164),   # Oklahoma City, OK
    "Southwest":     (35.0844, -106.6504),  # Albuquerque, NM
    "Northwest":     (47.6062, -122.3321),  # Seattle, WA
    "West":          (37.630589, -119.032822)  # Mammoth, CA
}

tree = cKDTree(np.column_stack([ds_obs.latitude.values,
                                ds_obs.longitude.values]))

target_coords = np.array(list(REGION_TARGET_POINTS.values()))  # (9, 2)
_, space_indices = tree.query(target_coords)

ds_obs = ds_obs.isel(space=space_indices)

In [ ]:
# Subset HRRR to the same points as the MADIS observations, and select only
# the variables of interest, converting temperature to Celsius.
hrrr_subset = subset_grid_to_point_xy(ds=hrrr, point_ds=ds_obs)
hrrr_subset= hrrr_subset[['ws_10', 'winddir_10', 'gust_2m', 't2m', 'r2']]
hrrr_subset['t2m'] = hrrr_subset['t2m'] - 273.15 #convert to Celsius

In [ ]:
# Remove coords that are no longer needed
hrrr_subset = hrrr_subset.drop_vars(['gribfile_projection', 'step', 'surface'])
ds_all = forecast_obs_merge(ds1=hrrr_subset,
                            ds2=ds_obs,
                            ds1_timevar="valid_time",
                            ds2_timevar="time",
                            ds1_prefix="hrrr")

In [ ]:
# Replace observed u and v wind components with wind speed and direction
ds_all["winddirection_10m"] = calc_winddirection(
    wind_v = ds_all['v_10m'],
    wind_u=ds_all['u_10m'],
)

ds_all["windspeed_10m"] = calc_windspeed(
    wind_v=ds_all['v_10m'],
    wind_u=ds_all['u_10m'],
)

ds_all = ds_all.drop_vars(['u_10m', 'v_10m'])

In [ ]:
# Add absolute difference variables to the dataset
for obs_var, model_var in VAR_NAME_MAP.items():
    ds_all = add_abs_diff(ds_all, var1=obs_var, var2=model_var)

## Applying quality control filters

In [ ]:
states = gpd.read_file(shape_file_path)
#reproject to wgs 84 for matching with hrrrnd madis lat/lon
states = states.to_crs('EPSG:4326')

In [ ]:
# Subset dataset to variables of interest
# Remove forecast variables from the dataset, retaining the
# observed variables and the absolute difference variables.
ds_all = ds_all[['abs_diff_temperature_2m', "temperature_2m", 'windspeed_10m',
                 'winddirection_10m', 'windgust_10m', 'temperature_2m',
                 'relativehumidity_2m', 'abs_diff_windspeed_10m',
                 'abs_diff_winddirection_10m', 'abs_diff_windgust_10m',
                 'abs_diff_relativehumidity_2m']]

In [ ]:
# Compute dask arrays into memory, reset space coordinate to integer index
ds = ds_all.copy()
ds = ds.compute()
ds = ds.assign_coords(space=np.arange(len(ds.space)))
ds_cleaned = ds.copy()

In [ ]:
# Parallelize and run the quality control process by region

region_polygons = {
    region: states[states["STUSPS"].isin(state_list)].unary_union
    for region, state_list in CLIMATE_REGIONS.items()
}

region_args = [
    (get_region_ds(ds, polygon), CLEAN_ORDER, CONFIG_MAP, MASK_DEPENDENCIES)
    for polygon in region_polygons.values()
]

cleaned_region_datasets = []
with ProcessPoolExecutor(max_workers=len(CLIMATE_REGIONS)) as executor:
    futures = [
        executor.submit(process_region, *args)
        for args in region_args
    ]
    for future in as_completed(futures):
        ds_region_cleaned = future.result()
        cleaned_region_datasets.append(ds_region_cleaned)

ds_cleaned = xr.concat(cleaned_region_datasets, dim="space")

## Cleaning and saving dataset

In [ ]:
# Remove absolute difference variables from the cleaned dataset,
# they are only needed for QC.
abs_diff_vars = [v for v in ds_cleaned.data_vars if v.startswith('abs_diff_')]
ds_cleaned = ds_cleaned.drop_vars(abs_diff_vars)

In [ ]:
# Drop stations with all NaN values after QC for all variables
valid_space_mask = ~ds_cleaned.to_array().isnull().all(
    dim=['time', 'variable'])
ds_cleaned = ds_cleaned.isel(space=valid_space_mask)

In [ ]:
# Convert space coordinate to lat/lon string
# (easier to merge across years than multi-index, retains trailing zeros)
lat_lon_strings = [f"{lat:.6f}_{lon:.6f}" for lat,
                   lon in zip(ds_cleaned.latitude.values,
                              ds_cleaned.longitude.values)]
ds_cleaned = ds_cleaned.assign_coords(space=lat_lon_strings)

In [ ]:
# Convert everything to float32
for var in ds_cleaned.data_vars:
    if ds_cleaned[var].dtype == np.float64:
        ds_cleaned[var] = ds_cleaned[var].astype(np.float32)

for coord in ds_cleaned.coords:
    if ds_cleaned[coord].dtype == np.float64:
        ds_cleaned[coord] = ds_cleaned[coord].astype(np.float32)

In [ ]:
# Add standard_name attributes to variables
standard_name = {
    "windspeed_10m": "Wind Speed",
    "winddirection_10m": "Wind Direction",
    "windgust_10m": "Wind Gust",
    "temperature_2m": "Temperature",
    "relativehumidity_2m": "Relative Humidity",
    "longitude": "Longitude",
    "latitude": "Latitude",
    "elevation": "Elevation"
}

for var, name in standard_name.items():
    if var in ds_cleaned:
        ds_cleaned[var].attrs["standard_name"] = name

In [ ]:
# Add units attributes to variables
units = {
    "windspeed_10m": "m s-1",
    "winddirection_10m": "degrees from north",
    "windgust_10m": "m s-1",
    "temperature_2m": "Degrees Celsius",
    "relativehumidity_2m": "%",
    "longitude": "degree_east",
    "latitude": "degree_north",
    'elevation': "m"
}

for var, unit in units.items():
    if var in ds_cleaned:
        ds_cleaned[var].attrs["units"] = unit

In [ ]:
# Chunk and save the cleaned dataset to a NetCDF file
ds_cleaned.chunk({'time': 1000, 'space': 5000}).to_netcdf(
    output_path,
    engine='h5netcdf',
)